<a href="https://colab.research.google.com/github/sukhan220/nlp_bangla_notebook/blob/main/Sequences_and_padded.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  টেক্সট থেকে সিকোয়েন্স ও প্যাডিং শেখা


---
### সিকোয়েন্স কি?  
মানুষ যখন কোনো বাক্য পড়ে বা শোনে, তখন শুধু আলাদা আলাদা শব্দ দেখে অর্থ বোঝে না। বরং শব্দগুলো কোন ক্রমে আছে এবং বাক্যে কে কী কাজ করছে, সেটাও বিবেচনা করে।

যেমন:

> **রাহুল হাসানকে ডাকল।**  
> **হাসান রাহুলকে ডাকল।**

উপরের প্রথম বাক্যে আমরা পড়ে বা শুনে বুঝতে পারি, রাহুল ডাকছে এবং কাকে ডাকছে? হাসানকে। আবার দ্বিতীয় বাক্যে হাসান ডাকছে এবং রাহুলকে ডাকা হচ্ছে। দুই বাক্যে একই শব্দ থাকলেও কাজ ভিন্ন।

তাহলে কি বলা যায়? টোকেনাইজেশনে যেভাবে দেখেছি, অক্ষর দিয়ে মেশিনকে বুঝানোর চেয়ে শব্দ দিয়ে টোকেন করে নেওয়া বা আইডি করে নেওয়া সহজ। সেভাবে ন্যাচারাল ল্যাঙ্গুয়েজ প্রসেসিং-এ বাক্যের একটা করে **ক্রম** বলে দিলে মেশিনকে ভাষা শেখানোটা আরও সহজ হয়।

টোকেনাইজেশনের পর প্রতিটি শব্দের জন্য একটি করে **ইউনিক ID** পেয়েছি আমরা। এখন মেশিনকে শুধু শব্দের ID দিলেই হবে না, বাক্যে কোন শব্দের পরে কোন শব্দ এসেছে এবং শব্দগুলোর মধ্যে কী সম্পর্ক তৈরি হয়েছে, সেটাও বুঝতে হবে। অর্থাৎ, মেশিনকে শব্দের ক্রম বা **সিকেয়েন্স** শেখাতে হবে।

ধরা যাক, আমাদের ডিকশনারিতে যে শব্দগুলো আছে, তাদের ID হলো:

| শব্দ     |   ID   |
| :------- | :----: |
| `আমি`    |  **1** |
| `বই`     |  **2** |
| `পছন্দ`  |  **3** |
| `করি`    |  **4** |
| `নতুন`   |  **5** |
| `পড়তে`   |  **6** |
| `লিখতে`  |  **7** |
| `লেখকের` |  **8** |
| `মাঝে`   |  **9** |
| `এআই`    | **10** |
| `একটা`   | **11** |

এখন বাক্যটি যদি হয়:

> **আমি বই পড়তে পছন্দ করি।**

তাহলে শব্দগুলোর আইডি গুলোকে **বাক্যের ক্রম অনুযায়ী** সাজালে পাওয়া যাবে:

```text
[1, 2, 6, 3, 4]
```

এই সংখ্যাগুলোর ধারাকেই বলা হচ্ছে **সিকয়েন্স**।

### শুরুটা করবো যতিচিহ্ন ও স্টপওয়ার্ড বাদ দিয়ে!

NLP তে টেক্সট নিয়ে কাজ করার সময় শুরুতেই কিছু অপ্রয়োজনীয় অংশ পরিষ্কার করে নেওয়া ভালো। এর মধ্যে **যতিচিহ্ন (Punctuation)** এবং **স্টপওয়ার্ড** ।

* **যতিচিহ্ন:** `।`, `,`, `.`, `!`, `?` ইত্যাদি চিহ্ন।
* **স্টপওয়ার্ড:** এমন কিছু সাধারণ শব্দ, যেগুলো বাক্যে বারবার ব্যবহৃত হলেও অনেক ক্ষেত্রে মূল অর্থ বোঝার জন্য খুব বেশি গুরুত্বপূর্ণ নয়। যেমনঃ `এবং`, `কিন্তু` ইত্যাদি।


In [49]:


from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer = Tokenizer(num_words=10)
sentences =[
    "আমি বই পড়তে পছন্দ করি!",
    "আমি বই লিখতে পছন্দ করি।"
]
tokenizer.fit_on_texts(sentences)
word_index= tokenizer.word_index
print(word_index)

{'আমি': 1, 'বই': 2, 'পছন্দ': 3, 'পড়তে': 4, 'করি': 5, 'লিখতে': 6, 'করি।': 7}


আমরা আগের `Tokenizer` এর কোডে ফিল্টার ব্যবহার করে ইংরেজি এক্সক্লেমেশন মার্ক (`!`) চিহ্নটি বাদ দিয়েছিলাম। তাই `করি` এবং `করি!`এই দুটিকে আলাদা টোকেন হিসেবে নেয়নি। অর্থাৎ, `!` চিহ্নটি বাদ দিয়ে `করি`কে একটি শব্দ হিসেবেই ধরেছিল।

কিন্তু এবার আমরা কোনো ফিল্টার না করেও দেখছি, `!` চিহ্নটি বাদ পড়ে যাচ্ছে। অথচ বাংলা যতিচিহ্ন দাঁড়ি (`।`) বাদ পড়ছে না। ফলে `করি।`কে একটি সম্পূর্ণ শব্দ হিসেবেই টোকেনাইজ করা হচ্ছে।

এখন প্রশ্ন হলো **এমনটা কেন হচ্ছে?**

এর কারণটা বুঝতে আমরা `get_config()` ব্যবহার করে একটু দেখে নিই।


In [50]:
tokenizer.get_config()

{'num_words': 10,
 'filters': '!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n',
 'lower': True,
 'split': ' ',
 'char_level': False,
 'oov_token': None,
 'document_count': 2,
 'word_counts': '{"\\u0986\\u09ae\\u09bf": 2, "\\u09ac\\u0987": 2, "\\u09aa\\u09dc\\u09a4\\u09c7": 1, "\\u09aa\\u099b\\u09a8\\u09cd\\u09a6": 2, "\\u0995\\u09b0\\u09bf": 1, "\\u09b2\\u09bf\\u0996\\u09a4\\u09c7": 1, "\\u0995\\u09b0\\u09bf\\u0964": 1}',
 'word_docs': '{"\\u09ac\\u0987": 2, "\\u09aa\\u099b\\u09a8\\u09cd\\u09a6": 2, "\\u0986\\u09ae\\u09bf": 2, "\\u0995\\u09b0\\u09bf": 1, "\\u09aa\\u09dc\\u09a4\\u09c7": 1, "\\u0995\\u09b0\\u09bf\\u0964": 1, "\\u09b2\\u09bf\\u0996\\u09a4\\u09c7": 1}',
 'index_docs': '{"2": 2, "3": 2, "1": 2, "5": 1, "4": 1, "7": 1, "6": 1}',
 'index_word': '{"1": "\\u0986\\u09ae\\u09bf", "2": "\\u09ac\\u0987", "3": "\\u09aa\\u099b\\u09a8\\u09cd\\u09a6", "4": "\\u09aa\\u09dc\\u09a4\\u09c7", "5": "\\u0995\\u09b0\\u09bf", "6": "\\u09b2\\u09bf\\u0996\\u09a4\\u09c7", "7": "\\u0995\\u09b0\\u09bf\\u0964"

আউটপুটে দেখা যাচ্ছে:

```text
'filters': '!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n'
```

তাহলে বলা যায়, **ফিল্টারের আগে থেকেই ইংরেজি এক্সক্লেমেশন মার্ক (`!`) সহ প্রচুর যতিচিহ্ন Keras ডিফল্টভাবে ফিল্টার করে রাখে।** এর মানে হলো, **দাঁড়ি (`।`) ডিফল্টভাবে ফিল্টার করা নেই।**


In [51]:
from tensorflow.keras.preprocessing.text import Tokenizer

# filters-এর ভেতরে বাংলা দাড়ি (।) যুক্ত করা হয়েছে
tokenizer = Tokenizer(num_words=10, filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n।')

sentences = [
    "আমি বই পড়তে পছন্দ করি।",
    "আমি বই লিখতে পছন্দ করি"
]

tokenizer.fit_on_texts(sentences)
print(tokenizer.word_index)

{'আমি': 1, 'বই': 2, 'পছন্দ': 3, 'করি': 4, 'পড়তে': 5, 'লিখতে': 6}


```python
filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n।'
```

এখানে `filters` এর একেবারে শেষে বাংলা যতিচিহ্ন **দাঁড়ি (`।`)** যোগ করা হয়েছে।

এর ফলে দাড়ি `।` চিহ্নটিও ফিল্টার হয়ে যাবে। যেমন:

```text
করি
করি!
করি।
```

এখন `'!'` এবং `'।'` বাদ দেওয়ার পর প্রতিটি ক্ষেত্রেই মূল শব্দটি হবে:

```text
করি
```

ফলে Tokenizer **`'করি'` এবং `'করি।'` কে আলাদা আলাদা টোকেন হিসেবে না ধরে মাত্র একটি টোকেন** হিসেবে গণ্য করবে।


In [52]:
# filters-এর ভেতরে বাংলা দাড়ি (।) যুক্ত করা হয়েছে
tokenizer = Tokenizer(num_words=10, filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n।')
sentences = [
    "আমি বই পড়তে পছন্দ করি।",
    "আমি বই লিখতে পছন্দ করি",
    "নতুন নতুন লেখকের মাঝে এআই এর একটা প্রভাব দেখা যাচ্ছে।"
]
tokenizer.fit_on_texts(sentences)
word_index = tokenizer.word_index
sequence = tokenizer.texts_to_sequences(sentences)
print(" Word Index: ", word_index)
print(" Sequence: ", sequence)

 Word Index:  {'আমি': 1, 'বই': 2, 'পছন্দ': 3, 'করি': 4, 'নতুন': 5, 'পড়তে': 6, 'লিখতে': 7, 'লেখকের': 8, 'মাঝে': 9, 'এআই': 10, 'এর': 11, 'একটা': 12, 'প্রভাব': 13, 'দেখা': 14, 'যাচ্ছে': 15}
 Sequence:  [[1, 2, 6, 3, 4], [1, 2, 7, 3, 4], [5, 5, 8, 9]]


# টেক্সট থেকে সিকোয়েন্সে রূপান্তর


```
sequence = tokenizer.texts_to_sequences(sentences)
```
`texts_to_sequences()` মেথড টি আমাদের জন্য ডিকশনারি থেকে সিকোয়েন্স এ রূপান্তর করে দিচ্ছে


## **১. ডিকশনারি (`word_index`):**

| শব্দ | ID | শব্দ | ID | শব্দ | ID | শব্দ | ID |
| :--- | :---: | :--- | :---: | :--- | :---: | :--- | :---: |
| `'আমি'` | **1** | `'পছন্দ'` | **3** | `'নতুন'` | **5** | `'লিখতে'` | **7** |
| `'বই'` | **2** | `'করি'` | **4** | `'পড়তে'` | **6** | `'লেখকের'` | **8** |
| `'মাঝে'` | **9** | `'এআই'` | **10** | `'একটা'` | **11** | *অন্যান্য* | **12+** |


## **২. ইনপুট বাক্য থেকে সিকোয়েন্সে রূপান্তর:**

* **বাক্য ১:** `"আমি বই পড়তে পছন্দ করি"`  
  ➔ **সিকোয়েন্স:** `[1, 2, 6, 3, 4]`

* **বাক্য ২:** `"আমি বই লিখতে পছন্দ করি"`  
  ➔ **সিকোয়েন্স:** `[1, 2, 7, 3, 4]`

* **বাক্য ৩:** `"নতুন নতুন লেখকের মাঝে এআই এর  একটা প্রভাব দেখা যাচ্ছে"`  
  ➔ **সিকোয়েন্স:** `[5, 5, 8, 9]` *১০ বা তার বড় ইনডেক্স শব্দের টোকেন গুলো সিকোয়েন্স থেকে বাদ পড়েছে গিয়েছে। যেমনঃ `এআই: 10` সিকোয়েন্স হওয়ার কথা ছিলো  `[5, 5, 8, 9, 10, 11, 12, 13, 14, 15]`*

---

> **দেখার বিষয়:**
> * `Tokenizer(num_words=10)` দেওয়া থাকলে কেবল **top 9** শব্দ  যেমনঃ ইনডেক্স ১ থেকে ৯ সিকোয়েন্সে আসবে।
> * ৩ নম্বর বাক্যের `'এআই'`এর ইনডেক্স ১০ এবং `'এর'` শব্দের  ইনডেক্স ১১ হওয়ার কারণে, `num_words=10` ফিল্টারের শর্ত অনুযায়ী এগুলো সিকোয়েন্সে যুক্ত হয়নি।

`Tokenizer(num_words=20)` করে দেওয়া হয় তখন  **top 19** শব্দ  যেমনঃ ইনডেক্স ১ থেকে 19 সিকোয়েন্সে পাওয়া যাবে।  

**বাক্য ৩:** `"নতুন নতুন লেখকের মাঝে এআই এর একটা প্রভাব দেখা যাচ্ছে"`  
  ➔ **সিকোয়েন্স:** `[5, 5, 8, 9, 10, 11, 12, 13, 14, 15]` ঠিক ভাবে পাওয়া যাবে। নিচে দেখে নেয়া যাক তাহলে।

In [53]:
# filters-এর ভেতরে বাংলা দাড়ি (।) যুক্ত করা হয়েছে
tokenizer = Tokenizer(num_words=20, filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n।')
tokenizer.fit_on_texts(sentences)
word_index = tokenizer.word_index
sequence = tokenizer.texts_to_sequences(sentences)
print(" Word Index: ", word_index)
print(" Sequence: ", sequence)

 Word Index:  {'আমি': 1, 'বই': 2, 'পছন্দ': 3, 'করি': 4, 'নতুন': 5, 'পড়তে': 6, 'লিখতে': 7, 'লেখকের': 8, 'মাঝে': 9, 'এআই': 10, 'এর': 11, 'একটা': 12, 'প্রভাব': 13, 'দেখা': 14, 'যাচ্ছে': 15}
 Sequence:  [[1, 2, 6, 3, 4], [1, 2, 7, 3, 4], [5, 5, 8, 9, 10, 11, 12, 13, 14, 15]]




### প্যাডিং কি?

সব বাক্যের দৈর্ঘ্য সমান হয় না। কোনো বাক্য ছোট, আবার কোনোটি বড়। যেমন, একটি বাক্য

>**আমি বই পছন্দ করি।**

এটার সিকয়েন্স উপরের টোকেন অনুসারে

```text
[1, 2, 3, 4]
```

আরেকটি বাক্যের এবং হলো সিকোয়েন্স যদি হয়:

>**আমি বই লিখতে পছন্দ করি।**

```text
[1, 2, 7, 3, 4]
```

প্রথমটির সিকোয়েন্সে ৪টি আইডি এবং দ্বিতীয়টিতে ৫টি আইডি আছে। কিন্তু মেশিনের জন্য সব সিকোয়েন্সের দৈর্ঘ্য একই রাখা সুবিধাজনক। তাই ছোট সিকোয়েন্সের শেষে অতিরিক্ত `0` যোগ করে সেটিকে বড় সিকোয়েন্সের সমান দৈর্ঘ্যের করা হয়:

| বাক্য | সিকোয়েন্স |
|:---|:---:|
| **আমি বই পছন্দ করি** | `[1, 2, 3, 4, 0]` |
| **আমি বই লিখতে পছন্দ করি** | `[1, 2, 7, 3, 4]` |

এখানে যোগ করা অতিরিক্ত `0` কে **প্যাডিং** বলা হয়। অর্থাৎ, ছোট সিকোয়েন্সকে নির্দিষ্ট দৈর্ঘ্যের করার জন্য অতিরিক্ত মান যোগ করার প্রক্রিয়াই হলো **প্যাডিং**।

In [54]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

padded = pad_sequences(sequence)

print(" Word Index: ", word_index)
print(" Sequence: ", sequence)
print(" Padded Sequence: ")
print(padded)

 Word Index:  {'আমি': 1, 'বই': 2, 'পছন্দ': 3, 'করি': 4, 'নতুন': 5, 'পড়তে': 6, 'লিখতে': 7, 'লেখকের': 8, 'মাঝে': 9, 'এআই': 10, 'এর': 11, 'একটা': 12, 'প্রভাব': 13, 'দেখা': 14, 'যাচ্ছে': 15}
 Sequence:  [[1, 2, 6, 3, 4], [1, 2, 7, 3, 4], [5, 5, 8, 9, 10, 11, 12, 13, 14, 15]]
 Padded Sequence: 
[[ 0  0  0  0  0  1  2  6  3  4]
 [ 0  0  0  0  0  1  2  7  3  4]
 [ 5  5  8  9 10 11 12 13 14 15]]


এখানে `pad_sequences()` মেথড ব্যবহার করে বিভিন্ন দৈর্ঘ্যের **সিকোয়েন্সকে একই দৈর্ঘ্যে** আনা হয়েছে।

```python
padded = pad_sequences(sequence)
```

| নং | মূল বাক্য (Original Sentence) | মূল সিকোয়েন্স (Sequence) | প্যাডেড সিকোয়েন্স (Padded Sequence) |
| :-: | :--- | :--- | :--- |
| **১** | `"আমি বই পড়তে পছন্দ করি।"` | `[1, 2, 6, 3, 4]` | `[0, 0, 0, 0, 0, 1, 2, 6, 3, 4]` |
| **২** | `"আমি বই লিখতে পছন্দ করি"` | `[1, 2, 7, 3, 4]` | `[0, 0, 0, 0, 0, 1, 2, 7, 3, 4]` |
| **৩** | `"নতুন নতুন লেখকের মাঝে এআই এর একটা প্রভাব দেখা যাচ্ছে।"` | `[5, 5, 8, 9, 10, 11, 12, 13, 14, 15]` | `[5, 5, 8, 9, 10, 11, 12, 13, 14, 15]` |

দেখা যাচ্ছে প্যাডিং লেন্থ বলে না দিও সবচেয়ে বড় বাক্যের শব্দ সংখ্যা ১০  `pad_sequences()` মেথড ডিফল্ড ভাবে ১০ লেন্থ নিয়েছে।

যদি `0` গুলো সিকোয়েন্সের শেষে এবং লেন্থ ধরে দিলে তাহলে কি হবে একটু দেখে নেয়া যায়।

In [55]:
padded = pad_sequences(sequence,maxlen=7,padding='post')

print(" Word Index: ", word_index)
print(" Sequence: ", sequence)
print(" Padded Sequence: ")
print(padded)

 Word Index:  {'আমি': 1, 'বই': 2, 'পছন্দ': 3, 'করি': 4, 'নতুন': 5, 'পড়তে': 6, 'লিখতে': 7, 'লেখকের': 8, 'মাঝে': 9, 'এআই': 10, 'এর': 11, 'একটা': 12, 'প্রভাব': 13, 'দেখা': 14, 'যাচ্ছে': 15}
 Sequence:  [[1, 2, 6, 3, 4], [1, 2, 7, 3, 4], [5, 5, 8, 9, 10, 11, 12, 13, 14, 15]]
 Padded Sequence: 
[[ 1  2  6  3  4  0  0]
 [ 1  2  7  3  4  0  0]
 [ 9 10 11 12 13 14 15]]


```python
padded = pad_sequences(sequence,maxlen=7,padding='post')
```

আমাদের টেবল যেমন ছিলোঃ

| নং | মূল বাক্য (Original Sentence) | মূল সিকোয়েন্স (Sequence) | প্যাডেড সিকোয়েন্স (Padded Sequence) |
| :-: | :--- | :--- | :--- |
| **১** | `"আমি বই পড়তে পছন্দ করি।"` | `[1, 2, 6, 3, 4]` | `[0, 0, 0, 0, 0, 1, 2, 6, 3, 4]` |
| **২** | `"আমি বই লিখতে পছন্দ করি"` | `[1, 2, 7, 3, 4]` | `[0, 0, 0, 0, 0, 1, 2, 7, 3, 4]` |
| **৩** | `"নতুন নতুন লেখকের মাঝে এআই এর একটা প্রভাব দেখা যাচ্ছে।"` | `[5, 5, 8, 9, 10, 11, 12, 13, 14, 15]` | `[5, 5, 8, 9, 10, 11, 12, 13, 14, 15]` |

এখন যেমন হয়েছেঃ
| নং | মূল বাক্য (Original Sentence) | মূল সিকোয়েন্স (Sequence) | প্যাডেড সিকোয়েন্স (Padded Sequence) |
| :-: | :--- | :--- | :--- |
| **১** | `"আমি বই পড়তে পছন্দ করি।"` | `[1, 2, 6, 3, 4]` | `[ 1, 2, 6, 3, 4, 0, 0]` |
| **২** | `"আমি বই লিখতে পছন্দ করি"` | `[1, 2, 7, 3, 4]` | `[ 1, 2, 7, 3, 4, 0, 0]` |
| **৩** | `"নতুন নতুন লেখকের মাঝে এআই এর একটা প্রভাব দেখা যাচ্ছে।"` | `[5, 5, 8, 9, 10, 11, 12, 13, 14, 15]` | `[ 9, 10, 11, 12, 13, 14, 15]` |

`maxlen=7` করাতে ১০ লেন্থের বাক্যের সিকোয়েন্সেরও ৭ দেখাচ্ছে প্যাডিং। `padding='post'` করাতে `0` সবশেষে এবং ১০ দৈঘ্যের সিকোয়েন্স শুরুতে কেটে শেষের দিক (`[ 9, 10, 11, 12, 13, 14, 15]`) থেকে ৭ টা নিয়েছে।

In [56]:
padded = pad_sequences(sequence,maxlen=3,padding='post', truncating='post')

print(" Word Index: ", word_index)
print(" Sequence: ", sequence)
print(" Padded Sequence: ")
print(padded)

 Word Index:  {'আমি': 1, 'বই': 2, 'পছন্দ': 3, 'করি': 4, 'নতুন': 5, 'পড়তে': 6, 'লিখতে': 7, 'লেখকের': 8, 'মাঝে': 9, 'এআই': 10, 'এর': 11, 'একটা': 12, 'প্রভাব': 13, 'দেখা': 14, 'যাচ্ছে': 15}
 Sequence:  [[1, 2, 6, 3, 4], [1, 2, 7, 3, 4], [5, 5, 8, 9, 10, 11, 12, 13, 14, 15]]
 Padded Sequence: 
[[1 2 6]
 [1 2 7]
 [5 5 8]]



```python
padded = pad_sequences(
    sequence,
    maxlen=7,
    padding='post',
    truncating='post'  # শেষের অংশ কেটে ফেলবে
)
```
### `truncating='post'`

ডিফল্টভাবে Keras এ `truncating='pre'` সেট করা থাকে, অর্থাৎ বাক্যের দৈর্ঘ্য `maxlen`এর চেয়ে বড় হলে **শুরুর দিক থেকে** শব্দ কেটে বাদ দেওয়া হয়। কিন্তু যখন আমরা **`truncating='post'`** সেট করবেন, তখন বাক্য বড় হলে **শেষের দিক থেকে** বাড়তি শব্দ কেটে বাদ দেওয়া হবে। যেমন এখানেঃ `maxlen=3` করাতে সবগুলো শেষের অংশ কেটে বাদ দেওয়া হয়ছে প্যাডিং সিকোয়েন্সে।
